# SDXL → GCS（互動式直接授權路徑）
選擇全新 NVIDIA L4 GPU。此 Notebook 尚未在 Jupyter 端到端驗證。此路徑與主機 CLI 管理路徑不同：授權僅由使用者在此互動式環境完成，不由主機傳入憑證。
總使用預算 20 分鐘，含安裝、下載、重試；請自行監看，完成或失敗下載後立刻刪除執行階段。這不是自動停止保證。

In [ ]:
from pathlib import Path
import json, sys, subprocess, uuid, time
WORK = Path("/content/sdxl-lifecycle")
WORK.mkdir(exist_ok=True)
SOURCES = {'generate.py': '"""One pinned SDXL image; imports are stdlib-only until run()."""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport os\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport zipfile\nfrom pathlib import Path\n\nMODEL_ID = \'stabilityai/stable-diffusion-xl-base-1.0\'\nMODEL_REVISION = \'462165984030d82259a11f4367a4eed129e94a7b\'\nPARAMETERS = dict(model=MODEL_ID, revision=MODEL_REVISION, variant=\'fp16\',\n    prompt=\'A small wooden cabin beside a quiet mountain lake at sunrise, watercolor illustration, warm light, no text\',\n    width=1024, height=1024, num_inference_steps=25, guidance_scale=7.5, seed=42)\n\ndef write_json(path, data):\n    path.write_text(json.dumps(data, indent=2, ensure_ascii=False) + \'\\n\')\n\ndef gpu_preflight():\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required; CPU fallback forbidden\')\n    name = torch.cuda.get_device_name(0)\n    if \'L4\' not in name.split():\n        raise RuntimeError(\'NVIDIA L4 required, got \' + name)\n    return torch\n\ndef run(output, preflight=gpu_preflight):\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=True)\n    if any(out.iterdir()):\n        raise FileExistsError(\'Output must be empty: \' + str(out))\n    start = time.monotonic()\n    metrics = dict(status=\'running\', python=sys.version, platform=platform.platform())\n    write_json(out/\'parameters.json\', PARAMETERS)\n    try:\n        # Select the PyTorch backend before importing transformers/diffusers;\n        # TensorFlow/Flax are not part of this pipeline and are not initialized.\n        os.environ[\'USE_TF\'] = \'0\'\n        os.environ[\'USE_FLAX\'] = \'0\'\n        torch = preflight()\n        from diffusers import StableDiffusionXLPipeline\n        import importlib.metadata\n        metrics.update(torch_version=torch.__version__, cuda_version=torch.version.cuda,\n            gpu=torch.cuda.get_device_name(0), packages={p: importlib.metadata.version(p) for p in\n                [\'diffusers\',\'transformers\',\'accelerate\',\'safetensors\',\'Pillow\',\'huggingface-hub\']})\n        try:\n            metrics[\'nvidia_smi_before\'] = subprocess.check_output([\'nvidia-smi\'], text=True, timeout=10)\n        except (OSError, subprocess.SubprocessError) as e:\n            metrics[\'nvidia_smi_error\'] = str(e)\n        torch.cuda.reset_peak_memory_stats()\n        load_start = time.monotonic()\n        pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_ID, revision=MODEL_REVISION,\n            torch_dtype=torch.float16, variant=\'fp16\', use_safetensors=True).to(\'cuda\')\n        # Keep diffusers\' standard force_upcast VAE behavior; no extra optimizations/refiner.\n        metrics[\'load_seconds\'] = time.monotonic()-load_start\n        torch.cuda.synchronize()\n        infer_start = time.monotonic()\n        image = pipe(prompt=PARAMETERS[\'prompt\'], width=1024, height=1024,\n            num_inference_steps=25, guidance_scale=7.5,\n            generator=torch.Generator(device=\'cuda\').manual_seed(42)).images[0]\n        torch.cuda.synchronize()\n        metrics[\'inference_seconds\'] = time.monotonic()-infer_start\n        metrics[\'peak_allocated_bytes\'] = torch.cuda.max_memory_allocated()\n        metrics[\'peak_reserved_bytes\'] = torch.cuda.max_memory_reserved()\n        if image.size != (1024,1024):\n            raise RuntimeError(\'Unexpected image dimensions\')\n        if all(lo == hi for lo,hi in image.convert(\'RGB\').getextrema()):\n            raise RuntimeError(\'Constant image rejected\')\n        image.save(out/\'image.png\')\n        metrics.update(status=\'success\', image_dimensions=list(image.size), image_nonconstant=True)\n    except BaseException as e:\n        metrics.update(status=\'failed\', error_type=type(e).__name__, error=str(e), traceback=traceback.format_exc())\n        raise\n    finally:\n        metrics[\'elapsed_seconds\'] = time.monotonic()-start\n        write_json(out/\'metrics.json\', metrics)\n        files = {p.name: dict(sha256=hashlib.sha256(p.read_bytes()).hexdigest(), bytes=p.stat().st_size)\n                 for p in sorted(out.iterdir()) if p.is_file() and p.name not in (\'manifest.json\',\'result.zip\')}\n        write_json(out/\'manifest.json\', dict(status=metrics[\'status\'], model=MODEL_ID,\n            resolved_revision=MODEL_REVISION, files=files))\n        with zipfile.ZipFile(out/\'result.zip\', \'w\', zipfile.ZIP_DEFLATED) as z:\n            for p in sorted(out.iterdir()):\n                if p.is_file() and p.name != \'result.zip\': z.write(p,p.name)\n    return metrics\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--output\', required=True)\n    run(parser.parse_args().output)\n', 'gcs_store.py': '"""Immutable generation-verified GCS REST uploads; credentials never serialized."""\nimport hashlib\nimport json\nimport re\nimport time\nfrom pathlib import Path\nfrom urllib.parse import quote\n\nTRANSIENT = {429,500,502,503,504}\n\ndef safe_path(value):\n    return bool(value and all(re.fullmatch(r\'[A-Za-z0-9][A-Za-z0-9._-]*\', p) and p not in (\'.\',\'..\') for p in value.split(\'/\')))\n\ndef is_timeout(exc):\n    return isinstance(exc, TimeoutError) or any(c.__name__ in (\'Timeout\',\'ReadTimeout\',\'ConnectTimeout\') for c in type(exc).__mro__)\n\nclass GCSStore:\n    def __init__(self, bucket, prefix, session=None):\n        prefix = prefix.rstrip(\'/\')\n        if not safe_path(prefix) or not re.fullmatch(r\'[a-z0-9][a-z0-9._-]{1,220}[a-z0-9]\',bucket):\n            raise ValueError(\'Supply a bucket name and safe nonempty relative prefix\')\n        self.bucket, self.prefix = bucket, prefix\n        if session is None:\n            import google.auth\n            from google.auth.transport.requests import AuthorizedSession\n            credentials, _ = google.auth.default(scopes=[\'https://www.googleapis.com/auth/cloud-platform\'])\n            session = AuthorizedSession(credentials)\n        self.session = session\n\n    def _get(self, url, params):\n        for attempt in range(3):\n            try:\n                r = self.session.get(url, params=params, timeout=60)\n                if r.status_code not in TRANSIENT: return r\n            except Exception as e:\n                if not is_timeout(e): raise\n                if attempt == 2: raise\n            if attempt < 2: time.sleep(0.25*2**attempt)\n        raise RuntimeError(\'GET transient retry budget exhausted\')\n\n    def put_verified(self, name, data, content_type=\'application/octet-stream\'):\n        if not safe_path(name): raise ValueError(\'Unsafe object name\')\n        obj = self.prefix+\'/\'+name\n        base = \'https://storage.googleapis.com\'\n        url = base+\'/storage/v1/b/\'+quote(self.bucket,safe=\'\')+\'/o/\'+quote(obj,safe=\'\')\n        generation = None\n        for attempt in range(3):\n            try:\n                r = self.session.post(base+\'/upload/storage/v1/b/\'+quote(self.bucket,safe=\'\')+\'/o\',\n                    params={\'uploadType\':\'media\',\'name\':obj,\'ifGenerationMatch\':\'0\'}, data=data,\n                    headers={\'Content-Type\':content_type}, timeout=60)\n            except Exception as e:\n                if not is_timeout(e): raise\n                r = None\n            if r is not None and r.status_code in (200,201):\n                generation = str(r.json()[\'generation\'])\n                break\n            if r is not None and r.status_code not in TRANSIENT:\n                raise RuntimeError(\'Immutable upload rejected HTTP \'+str(r.status_code))\n            # Ambiguous commit: inspect existing generation before retrying create-only.\n            metadata = self._get(url,{})\n            if metadata.status_code == 200:\n                generation = str(metadata.json()[\'generation\'])\n                break\n            if metadata.status_code != 404: raise RuntimeError(\'Recovery metadata HTTP \'+str(metadata.status_code))\n            if attempt < 2: time.sleep(0.25*2**attempt)\n        if generation is None: raise RuntimeError(\'Upload transient retry budget exhausted\')\n        read = self._get(url,{\'alt\':\'media\',\'generation\':generation})\n        sha = hashlib.sha256(data).hexdigest()\n        if read.status_code != 200 or hashlib.sha256(read.content).hexdigest() != sha:\n            raise RuntimeError(\'Generation readback checksum mismatch\')\n        return dict(object=obj,generation=generation,sha256=sha,bytes=len(data))\n\n    def publish(self, directory):\n        directory = Path(directory)\n        manifest = json.loads((directory/\'manifest.json\').read_text())\n        metrics = json.loads((directory/\'metrics.json\').read_text())\n        if manifest.get(\'status\') != \'success\' or metrics.get(\'status\') != \'success\':\n            raise ValueError(\'Cannot publish SUCCESS for a failed generation\')\n        required = [\'image.png\',\'parameters.json\',\'metrics.json\']\n        payloads = {n:(directory/n).read_bytes() for n in required}\n        for name,data in payloads.items():\n            entry = manifest.get(\'files\',{}).get(name,{})\n            if entry.get(\'sha256\') != hashlib.sha256(data).hexdigest() or entry.get(\'bytes\') != len(data):\n                raise ValueError(\'Local manifest mismatch: \'+name)\n        if (directory/\'result.zip\').exists(): payloads[\'result.zip\'] = (directory/\'result.zip\').read_bytes()\n        payloads[\'manifest.json\'] = (directory/\'manifest.json\').read_bytes()\n        objects = {n:self.put_verified(n,data,\'application/json\' if n.endswith(\'.json\') else \'application/octet-stream\') for n,data in payloads.items()}\n        success = json.dumps({\'status\':\'success\',\'objects\':objects},sort_keys=True).encode()\n        objects[\'SUCCESS.json\'] = self.put_verified(\'SUCCESS.json\',success,\'application/json\')\n        return dict(bucket=self.bucket,prefix=self.prefix,objects=objects)\n', 'requirements.txt': "# Reuse Colab's CUDA torch; intentionally do not install torch.\ndiffusers==0.35.1\ntransformers==4.55.4\naccelerate==1.11.0\nsafetensors==0.6.2\nPillow==11.3.0\nhuggingface-hub==0.36.0\ngoogle-auth==2.40.3\nrequests==2.32.5\n", 'bootstrap.py': '"""Install into an isolated environment, reusing only Colab\'s base packages."""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport urllib.request\nimport zipfile\n\nURL = \'https://files.pythonhosted.org/packages/a7/fe/a87be2a492440945e745b0dc81aef7fea9730d8194af88d854b7641f5ddf/uv-0.12.2-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl\'\nSHA256 = \'3a8d93d1fe019561b70494a8e1332492afb5f14dadc33d5239569d0964164d10\'\nENV = Path(\'/content/sdxl-env\')\n\n\ndef run(arguments):\n    env = dict(os.environ)\n    env.pop(\'UV_SYSTEM_PYTHON\', None)\n    result = subprocess.run(arguments, capture_output=True, text=True, env=env)\n    print(result.stdout, flush=True)\n    print(result.stderr, file=sys.stderr, flush=True)\n    if result.returncode:\n        raise RuntimeError(\'Dependency setup subprocess failed: \' + str(result.returncode))\n\n\ndef setup(requirements):\n    wheel = Path(\'/content/sdxl-uv.whl\')\n    urllib.request.urlretrieve(URL, wheel)\n    if hashlib.sha256(wheel.read_bytes()).hexdigest() != SHA256:\n        raise RuntimeError(\'uv wheel checksum mismatch\')\n    with zipfile.ZipFile(wheel) as archive:\n        members = [n for n in archive.namelist() if n.endswith(\'.data/scripts/uv\') or n == \'uv/uv\']\n        if len(members) != 1:\n            raise RuntimeError(\'Unexpected uv wheel layout\')\n        uv = Path(\'/content/sdxl-uv\')\n        uv.write_bytes(archive.read(members[0]))\n    uv.chmod(0o755)\n    run([str(uv), \'venv\', \'--system-site-packages\', \'--python\', sys.executable, str(ENV)])\n    # uv resolves target dependencies independently of inherited site-packages.\n    # Exclude base torch to avoid replacing its matching torchvision/CUDA stack.\n    import torch\n    base_torch = torch.__version__\n    exclude = Path(\'/content/sdxl-excludes.txt\')\n    exclude.write_text(\'torch\\n\')\n    run([str(uv), \'pip\', \'install\', \'--python\', str(ENV/\'bin/python\'), \'--excludes\', str(exclude), \'-r\', str(requirements)])\n    run([str(ENV/\'bin/python\'), \'-c\', \'import torch,torchvision; assert torch.__version__ == \'+repr(base_torch)+\'; print("Preserved base torch/torchvision:",torch.__version__,torchvision.__version__)\'])\n    Path(\'/content/sdxl-install.json\').write_text(json.dumps({\'returncode\': 0, \'isolated_environment\': str(ENV)}))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--requirements\', required=True)\n    setup(parser.parse_args().requirements)\n'}
for name, content in SOURCES.items():
    (WORK/name).write_text(content)
sys.path.insert(0,str(WORK))
started = time.monotonic()


In [ ]:
# Authenticate and prove create + exact-generation GET BEFORE installs/model load.
from google.colab import auth
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
from gcs_store import GCSStore
BUCKET = input('GCS bucket (name only, required): ').strip()
PREFIX = 'runs/sdxl-' + uuid.uuid4().hex
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
store = GCSStore(BUCKET, PREFIX, session=AuthorizedSession(credentials))
probe = store.put_verified('preflight.json', json.dumps({'probe':uuid.uuid4().hex}).encode(), 'application/json')
print('Verified preflight:', probe)
subprocess.run(['nvidia-smi'],check=True)
from generate import gpu_preflight
gpu_preflight()


In [ ]:
remaining = 1200 - (time.monotonic()-started)
if remaining <= 0: raise TimeoutError('20-minute budget exhausted')
subprocess.run([sys.executable, str(WORK/'bootstrap.py'), '--requirements', str(WORK/'requirements.txt')],check=True,timeout=remaining)


In [ ]:
OUTPUT = Path('/content/sdxl-output')
remaining = 1200 - (time.monotonic()-started)
if remaining <= 0: raise TimeoutError('20-minute budget exhausted')
generation_error = None
try:
    subprocess.run(['/content/sdxl-env/bin/python',str(WORK/'generate.py'),'--output',str(OUTPUT)],check=True,timeout=remaining)
except Exception as exc:
    generation_error = exc
    print('Generation failed; available artifacts remain for download:', repr(exc))
if (OUTPUT/'image.png').exists():
    from IPython.display import display, Image
    display(Image(filename=str(OUTPUT/'image.png')))


In [ ]:
# Download fallback runs even if publishing raises; no automatic runtime deletion.
from google.colab import files
try:
    if generation_error is not None: raise generation_error
    if time.monotonic()-started >= 1200: raise TimeoutError('20-minute budget exhausted; download now')
    receipt = store.publish(OUTPUT)
    (OUTPUT/'receipt.json').write_text(json.dumps(receipt,indent=2))
    print(json.dumps(receipt,indent=2))
finally:
    archive = OUTPUT/'result.zip'
    if not archive.exists() and OUTPUT.exists():
        import zipfile
        with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
            for f in OUTPUT.iterdir():
                if f.is_file() and f != archive: z.write(f,f.name)
    if archive.exists(): files.download(str(archive))
    else: print('No artifacts available; inspect earlier failure.')


## 必做：確認下載完成後刪除執行階段
先確認瀏覽器已完整下載 result.zip（以及 GCS receipt，如有）。然後使用 Colab「執行階段 → 中斷連線並刪除執行階段」。不要只關分頁。這份 Notebook 不宣稱自動 teardown；下載完成前不會呼叫解除分配。失敗、逾時也須執行這一步。